# Treinamento e Exportação TinyML (ESP32) — Sistema de Irrigação Inteligente (PIBEX 2026 - IFPE Paulista)

Este notebook implementa o pipeline completo da **Fase 1** planejada em `docs/ACTION_PLAN_AI_FIRMWARE.md` e `docs/DATASETS_EVALUATION.md` para execução direta no **Google Colab** (ou localmente via Jupyter).

### Arquitetura e Decisões Técnicas Implementadas:
1. **Dataset Oficial:** *Smart Agriculture Dataset (`cropdata_updated.csv`)* com `16.411` registros.
2. **Classificação Multiclasse de 3 Estados (`Softmax`):**
   - **`0 (OFF)`:** Umidade adequada -> Válvula fechada.
   - **`1 (IRRIGATE)`:** Necessidade hídrica -> Válvula aberta.
   - **`2 (THERMAL_STRESS_WET_SOIL)`:** Calor extremo (>32 °C) com solo já úmido (acima da capacidade de campo) -> Mantém válvula **fechada** (evita asfixia radicular) e emite **Alerta de Estresse Térmico** no log CSV/painel.
3. **Contexto Agronômico Opcional (*Context Dropout* com Token `0 = UNKNOWN`):**
   - **Entradas Obrigatórias (Sensores ESP32):** Temperatura do Ar (`DHT22`), Umidade do Ar (`DHT22`) e Umidade do Solo (`HW-080`).
   - **Entradas Opcionais (Configuração do Canteiro no `LittleFS`):** `soil_type` (Tipo de Solo), `crop ID` (Cultura, priorizando hortaliças locais: Tomate, Pimenta/Pimentão, Cenoura e Batata) e `Seedling Stage` (Estágio de Crescimento).
   - **Execução Não-Bloqueante:** As variáveis opcionais são codificadas via *One-Hot Encoding* onde o estado `0 (UNKNOWN / Não Informado)` é representado pelo **vetor totalmente nulo (`0.0`)**. Durante o treino, aplicamos *Context Dropout* (mascarando aleatoriamente o contexto parcial ou total) para que a rede neural opere com **alta precisão tanto com contexto completo (~99-100%) quanto apenas com os 3 sensores físicos (sem nenhum contexto informado)**.
4. **Exportação Direta para o ESP32:** Gera `model.tflite` (quantizado), valida no `tf.lite.Interpreter`, extrai *Golden Test Vectors* e exporta o cabeçalho C/C++ `model_data.h` pronto para o PlatformIO.

## 1. Preparação do Ambiente e Dependências

In [ ]:
# No Google Colab, TensorFlow, NumPy, Pandas, Scikit-Learn e Matplotlib já vêm instalados.
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

# Sementes fixas para total reprodutibilidade científica
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")

## 2. Download e Carregamento Automático do Dataset (`kagglehub`)
Baixa automaticamente a versão mais recente do dataset **`chaitanyagopidesi/smart-agriculture-dataset`** diretamente do Kaggle via `kagglehub` (com fallback para o arquivo local do repositório caso já esteja baixado).

In [ ]:
import glob
import kagglehub

# Verifica primeiro se o CSV já existe localmente no repositório; caso contrário, baixa via kagglehub
LOCAL_PATHS = [
    "datasets/Smart Agriculture Dataset/cropdata_updated.csv",
    "ai-model/datasets/Smart Agriculture Dataset/cropdata_updated.csv",
    "cropdata_updated.csv",
]

dataset_path = next((p for p in LOCAL_PATHS if os.path.exists(p)), None)

if dataset_path is None:
    # Download latest version directly from Kaggle via kagglehub
    path = kagglehub.dataset_download("chaitanyagopidesi/smart-agriculture-dataset")
    print("Path to dataset files:", path)
    csv_files = glob.glob(os.path.join(path, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em: {path}")
    dataset_path = csv_files[0]

df = pd.read_csv(dataset_path)
print(f"Dataset carregado de: {dataset_path}")
print(f"Formato: {df.shape[0]} linhas x {df.shape[1]} colunas | Valores nulos: {df.isnull().sum().sum()}")
display(df.head())

## 3. Mapeamento Agronômico, Normalização, Features Físicas (VPD) e *Context Dropout*

- Os **3 sensores físicos** são normalizados em `[0.0, 1.0]` (`TEMP: [0..50 °C]`, `HUMIDITY: [0..100 %]`, `SOIL_MOISTURE: [0..100 %]`).
- Adicionamos **4 features psicrométricas/agronômicas** (`VPD`, `EvapDemand`, `Temp/MOI`, `MOI*Temp`) calculáveis em `< 15 µs` no ESP32.
- Total: **27 features float32** (com proteção de sentinelas `-128 / -999 / 255` exportada para o C++).

In [ ]:
# Limites explícitos de normalização compartilhados entre Python e ESP32 (C++)
TEMP_MIN, TEMP_MAX = 0.0, 50.0
HUM_MIN, HUM_MAX = 0.0, 100.0
SOIL_MIN, SOIL_MAX = 0.0, 100.0

# Sentinelas de erro do hardware (SensorsManager.cpp) para proteção contra falha silenciosa
SENTINEL_TEMP_ERR_I8 = -128
SENTINEL_TEMP_ERR_I16 = -999
SENTINEL_HUM_ERR_U8 = 255

SOIL_TYPES = [
    "Clay Soil", "Sandy Soil", "Red Soil", "Loam Soil",
    "Black Soil", "Alluvial Soil", "Chalky Soil"
]
CROP_TYPES = ["Tomato", "Chilli", "Carrot", "Potato", "Wheat"]
SEEDLING_STAGES = [
    "Germination", "Seedling Stage",
    "Vegetative Growth / Root or Tuber Development",
    "Flowering", "Pollination", "Fruit/Grain/Bulb Formation",
    "Maturation", "Harvest"
]

SOIL_TO_ID = {name: i + 1 for i, name in enumerate(SOIL_TYPES)}
CROP_TO_ID = {name: i + 1 for i, name in enumerate(CROP_TYPES)}
STAGE_TO_ID = {name: i + 1 for i, name in enumerate(SEEDLING_STAGES)}

# 3 sensores + 7 solos + 5 culturas + 8 estágios + 4 features psicrométricas (VPD, EvapDemand, T/MOI, MOI*T)
INPUT_DIM = 3 + len(SOIL_TYPES) + len(CROP_TYPES) + len(SEEDLING_STAGES) + 4  # 27
NUM_CLASSES = 3

def compute_vpd_kpa(temp_c, rh_pct):
    es = 0.61078 * np.exp((17.27 * temp_c) / (temp_c + 237.3))
    return float(es * np.clip(1.0 - rh_pct / 100.0, 0.0, 1.0))

def encode_features(temp, hum, moi, soil_id=0, crop_id=0, stage_id=0):
    vec = np.zeros(INPUT_DIM, dtype=np.float32)
    t_n = float(np.clip((temp - TEMP_MIN) / (TEMP_MAX - TEMP_MIN), 0.0, 1.0))
    h_n = float(np.clip((hum - HUM_MIN) / (HUM_MAX - HUM_MIN), 0.0, 1.0))
    m_n = float(np.clip((moi - SOIL_MIN) / (SOIL_MAX - SOIL_MIN), 0.0, 1.0))
    vec[0], vec[1], vec[2] = t_n, h_n, m_n

    if 1 <= soil_id <= len(SOIL_TYPES):
        vec[3 + (soil_id - 1)] = 1.0
    if 1 <= crop_id <= len(CROP_TYPES):
        vec[3 + len(SOIL_TYPES) + (crop_id - 1)] = 1.0
    if 1 <= stage_id <= len(SEEDLING_STAGES):
        vec[3 + len(SOIL_TYPES) + len(CROP_TYPES) + (stage_id - 1)] = 1.0

    # Engenharia de Atributos Físicos (calculáveis em <15 us no ESP32)
    vpd = compute_vpd_kpa(temp, hum)
    vec[23] = float(np.clip(vpd / 8.0, 0.0, 1.0))
    vec[24] = float(np.clip((temp * (100.0 - hum)) / 4000.0, 0.0, 1.0))
    vec[25] = float(np.clip(t_n / (m_n + 0.10), 0.0, 5.0) / 5.0)
    vec[26] = float(m_n * t_n)
    return vec

print(f"Dimensão total do vetor de entrada (PhysMLP): {INPUT_DIM} features")

## 4. Deduplicação, Divisão Estratificada (70/15/15) e *Full-8 Mask Context Dropout*

1. Removemos as **128 duplicatas exatas** antes do split para evitar *data leakage*.
2. Aplicamos *Full-8 Mask Context Dropout* cobrindo todas as combinações de contexto (incluindo `Apenas Cultura Informada [0, C, 0]`).
3. Usamos pesos calibrados `{0: 1.0, 1: 1.15, 2: 2.20}` para preservar alto Recall na Classe 2 sem colapsar a Precisão no Modo Zero Contexto.

In [ ]:
# 1. Remoção obrigatória de duplicatas exatas antes do split (evita data leakage)
n_raw = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Deduplicação: {n_raw} -> {len(df)} linhas únicas ({n_raw - len(df)} duplicatas removidas)")

df["soil_id"] = df["soil_type"].map(SOIL_TO_ID)
df["crop_id"] = df["crop ID"].map(CROP_TO_ID)
df["stage_id"] = df["Seedling Stage"].map(STAGE_TO_ID)
df["strat_key"] = df["crop ID"].astype(str) + "_" + df["result"].astype(str)

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["strat_key"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["strat_key"]
)
print(f"Split Estratificado -> Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")

def build_full8_augmented_dataset(dataframe):
    """Gera todas as 8 combinações de máscara de contexto (Full-8 Mask Context Dropout)."""
    X_list, y_list = [], []
    for _, row in dataframe.iterrows():
        t, h, m = float(row["temp"]), float(row["humidity"]), float(row["MOI"])
        s_id, c_id, st_id = int(row["soil_id"]), int(row["crop_id"]), int(row["stage_id"])
        label = int(row["result"])
        is_vegetable = (row["crop ID"] != "Wheat")

        # Máscaras com Cultura informada (todas as culturas)
        for ms, mc, mst in [(1, 1, 1), (1, 1, 0), (0, 1, 1), (0, 1, 0)]:
            X_list.append(encode_features(t, h, m, s_id if ms else 0, c_id if mc else 0, st_id if mst else 0))
            y_list.append(label)

        # Máscaras sem Cultura informada (fallback seguro para hortaliças locais)
        if is_vegetable:
            for ms, mc, mst in [(1, 0, 1), (1, 0, 0), (0, 0, 1), (0, 0, 0)]:
                X_list.append(encode_features(t, h, m, s_id if ms else 0, 0, st_id if mst else 0))
                y_list.append(label)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)

X_train, y_train = build_full8_augmented_dataset(train_df)
X_val, y_val = build_full8_augmented_dataset(val_df)

# Pesos calibrados (evita colapso de Precisão na Classe 2 causado por class_weight=balanced ~5.05)
class_weight_dict = {0: 1.0, 1: 1.15, 2: 2.20}
print(f"Treino Full-8: {X_train.shape[0]} amostras | Validação: {X_val.shape[0]} amostras")
print(f"Pesos de Classe Calibrados: {class_weight_dict}")

## 5. Arquitetura da Rede Neural MLP (TinyML) e Treinamento

A arquitetura utiliza apenas camadas `Dense` com ativação `ReLU` e camada final `Softmax` de 3 neurônios (`AddFullyConnected`, `AddRelu`, `AddSoftmax` no TensorFlow Lite Micro), ocupando menos de **8 KB de Flash** e **4 KB de `tensor_arena` (RAM)** no ESP32.

In [ ]:
def build_tinyml_model(input_dim=INPUT_DIM, num_classes=NUM_CLASSES):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,), name="sensor_and_context_input"),
        tf.keras.layers.Dense(32, activation="relu", name="dense_1"),
        tf.keras.layers.Dense(16, activation="relu", name="dense_2"),
        tf.keras.layers.Dense(num_classes, activation="softmax", name="decision_output")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_tinyml_model()
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=12, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

## 6. Avaliação em Múltiplos Cenários de Operação do ESP32

In [ ]:
CLASS_NAMES = ["0: OFF (Nao Irrigar)", "1: IRRIGATE (Irrigar)", "2: THERMAL_STRESS (Alerta)"]

def evaluate_scenario(dataframe, mode_name, mask_fn):
    X_s, y_s = [], []
    for _, row in dataframe.iterrows():
        s_id, c_id, st_id = mask_fn(int(row["soil_id"]), int(row["crop_id"]), int(row["stage_id"]))
        X_s.append(encode_features(float(row["temp"]), float(row["humidity"]), float(row["MOI"]), s_id, c_id, st_id))
        y_s.append(int(row["result"]))
    X_s, y_s = np.array(X_s, dtype=np.float32), np.array(y_s, dtype=np.int32)
    preds = np.argmax(model.predict(X_s, verbose=0), axis=1)
    sep = "=" * 75
    print(f"\n{sep}\nAvaliação: {mode_name} (n={len(y_s)})\n{sep}")
    print(classification_report(y_s, preds, target_names=CLASS_NAMES, digits=4))
    return y_s, preds

y_true_full, y_pred_full = evaluate_scenario(
    test_df, "MODO 1: Contexto Completo (Sensores + Solo + Cultura + Estágio)",
    lambda s, c, st: (s, c, st)
)
_ = evaluate_scenario(
    test_df, "MODO 2: Canteiro Fixo (Sensores + Solo + Cultura | Estágio = UNKNOWN)",
    lambda s, c, st: (s, c, 0)
)
_ = evaluate_scenario(
    test_df, "MODO 6: Apenas Cultura Informada (Sensores + Cultura | Solo e Estágio = UNKNOWN)",
    lambda s, c, st: (0, c, 0)
)
veg_test_df = test_df[test_df["crop ID"] != "Wheat"]
y_true_zero, y_pred_zero = evaluate_scenario(
    veg_test_df, "MODO 8: Zero Contexto / Só 3 Sensores DHT22+HW080 (Hortaliças | Tudo = UNKNOWN)",
    lambda s, c, st: (0, 0, 0)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history.history["loss"], label="Treino Loss")
axes[0].plot(history.history["val_loss"], label="Validação Loss")
axes[0].set_title("Evolução da Loss (Crossentropy)")
axes[0].legend()
ConfusionMatrixDisplay.from_predictions(
    y_true_full, y_pred_full, display_labels=["OFF", "IRRIG", "STRESS"], ax=axes[1], cmap="Greens"
)
axes[1].set_title("Matriz de Confusão (Contexto Completo)")
ConfusionMatrixDisplay.from_predictions(
    y_true_zero, y_pred_zero, display_labels=["OFF", "IRRIG", "STRESS"], ax=axes[2], cmap="Blues"
)
axes[2].set_title("Matriz de Confusão (Zero Contexto)")
plt.tight_layout()
plt.show()

## 7. Conversão Real Full INT8 (`.tflite`), Validação Cruzada e Exportação de `model_data.h`

In [ ]:
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
tflite_path = os.path.join(OUTPUT_DIR, "model_int8.tflite")
header_path = os.path.join(OUTPUT_DIR, "model_data.h")

def representative_data_gen():
    for i in range(min(300, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

# Conversão Real Full Integer Quantization (INT8) para TensorFlow Lite Micro
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"[TinyML] Modelo INT8 salvo: {tflite_path} ({len(tflite_model)} bytes / {len(tflite_model)/1024:.2f} KB)")

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
in_scale, in_zp = input_details["quantization"]
out_scale, out_zp = output_details["quantization"]

def predict_tflite(x_vec):
    xq = np.round(np.expand_dims(x_vec, axis=0) / in_scale + in_zp).astype(np.int8)
    interpreter.set_tensor(input_details["index"], xq)
    interpreter.invoke()
    yq = interpreter.get_tensor(output_details["index"])[0]
    return (yq.astype(np.float32) - out_zp) * out_scale

max_diff = 0.0
agree = 0
n_check = min(300, len(X_val))
for i in range(n_check):
    p_keras = model.predict(np.expand_dims(X_val[i], axis=0), verbose=0)[0]
    p_tflite = predict_tflite(X_val[i])
    max_diff = max(max_diff, float(np.max(np.abs(p_keras - p_tflite))))
    if int(np.argmax(p_keras)) == int(np.argmax(p_tflite)):
        agree += 1

print(f"[Validação Cruzada INT8] Concordância: {100*agree/n_check:.2f}% | Max Diff: {max_diff:.6f}")

golden_cases = [
    ("Caso 1 (Sem Contexto - Solo Umido/Frio -> OFF)", 22.0, 78.0, 65.0, 0, 0, 0),
    ("Caso 2 (Sem Contexto - Solo Seco/Quente -> IRRIGATE)", 34.0, 45.0, 22.0, 0, 0, 0),
    ("Caso 3 (Sem Contexto - Calor Extremo + Solo Encharcado -> THERMAL_STRESS)", 38.0, 40.0, 75.0, 0, 0, 0),
    ("Caso 4 (Com Contexto: Tomate em Solo Argiloso + Floracao - Seco -> IRRIGATE)", 33.0, 50.0, 28.0, SOIL_TO_ID["Clay Soil"], CROP_TO_ID["Tomato"], STAGE_TO_ID["Flowering"]),
    ("Caso 5 (Com Contexto: Pimenta em Solo Arenoso + Germinacao - Calor + Umido -> THERMAL_STRESS)", 37.0, 42.0, 70.0, SOIL_TO_ID["Sandy Soil"], CROP_TO_ID["Chilli"], STAGE_TO_ID["Germination"]),
]

golden_comments = []
for desc, t, h, m, s_id, c_id, st_id in golden_cases:
    vec = encode_features(t, h, m, s_id, c_id, st_id)
    probs = predict_tflite(vec)
    pred_cls = int(np.argmax(probs))
    golden_comments.append(f"// {desc}: T={t}C, H={h}%, M={m}% => Class={pred_cls} (P=[{probs[0]:.3f},{probs[1]:.3f},{probs[2]:.3f}])")

hex_lines = ["  " + ", ".join(f"0x{b:02x}" for b in tflite_model[i:i+12]) for i in range(0, len(tflite_model), 12)]
hex_body = ",\n".join(hex_lines)
golden_block = "\n".join(golden_comments)

c_header = f"""#pragma once
#include <cstdint>

// Golden Test Vectors (TFLite Full INT8):
{golden_block}

namespace TinyMLConfig {{
    constexpr uint8_t INPUT_FEATURE_COUNT = {INPUT_DIM};
    constexpr uint8_t OUTPUT_CLASS_COUNT = {NUM_CLASSES};
    constexpr float TEMP_MIN = {TEMP_MIN}f;
    constexpr float TEMP_MAX = {TEMP_MAX}f;
    constexpr float HUMIDITY_MIN = {HUM_MIN}f;
    constexpr float HUMIDITY_MAX = {HUM_MAX}f;
    constexpr float SOIL_MIN = {SOIL_MIN}f;
    constexpr float SOIL_MAX = {SOIL_MAX}f;
    // Sentinelas de falha do DHT22 (SensorsManager.cpp) - abortar inferencia se detectados!
    constexpr int8_t SENTINEL_TEMP_ERROR_I8 = {SENTINEL_TEMP_ERR_I8};
    constexpr int16_t SENTINEL_TEMP_ERROR_I16 = {SENTINEL_TEMP_ERR_I16};
    constexpr uint8_t SENTINEL_HUM_ERROR_U8 = {SENTINEL_HUM_ERR_U8};
    constexpr float DEFAULT_CONFIDENCE_THRESHOLD = 0.55f;
}}

const unsigned int g_model_len = {len(tflite_model)};
alignas(8) const unsigned char g_model_data[] = {{
{hex_body}
}};
"""

with open(header_path, "w", encoding="utf-8") as f:
    f.write(c_header)
print(f"[TinyML] Header C++ exportado: {header_path}")